In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import matplotlib.pyplot as plt
import seaborn as sns

import joblib
import os

In [22]:
patients = pd.read_csv("patients_cleaned.csv")

patients.head()

,Patient_ID,Patient_Name,Age,Gender,Village,Visit_Date,Season,Disease,Medicine_Name,Quantity,Doctor,Year,Month
0,1,Patient_1,4,Female,Sonapur,2025-12-01,Winter,Fever,Paracetamol,1,Dr. Sharma,2025,12
1,2,Patient_2,64,Female,Lakshmi Nagar,2025-11-25,Post-Monsoon,Diarrhoea,ORS,10,Dr. Deshmukh,2025,11
2,3,Patient_3,2,Male,Rahatgaon,2026-05-19,Summer,Cough,Cough Syrup,1,Dr. Deshmukh,2026,5
3,4,Patient_4,41,Male,Bhavanipur,2025-05-26,Summer,Common Cold,Cetirizine,10,Dr. Singh,2025,5
4,5,Patient_5,33,Male,Khed,2025-03-12,Summer,Anaemia,Iron Tablets,3,Dr. Patil,2025,3


In [23]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Patient_ID     3000 non-null   int64 
 1   Patient_Name   3000 non-null   object
 2   Age            3000 non-null   int64 
 3   Gender         3000 non-null   object
 4   Village        3000 non-null   object
 5   Visit_Date     3000 non-null   object
 6   Season         3000 non-null   object
 7   Disease        3000 non-null   object
 8   Medicine_Name  3000 non-null   object
 9   Quantity       3000 non-null   int64 
 10  Doctor         3000 non-null   object
 11  Year           3000 non-null   int64 
 12  Month          3000 non-null   int64 
dtypes: int64(5), object(8)
memory usage: 304.8+ KB


In [24]:
patients["Visit_Date"] = pd.to_datetime(patients["Visit_Date"])

patients["Year"] = patients["Visit_Date"].dt.year

patients["Month"] = patients["Visit_Date"].dt.month

In [25]:
cases = (
    patients
    .groupby(
        [
            "Village",
            "Disease",
            "Season",
            "Year",
            "Month"
        ]
    )
    .size()
    .reset_index(name="Cases")
)

cases.head()

,Village,Disease,Season,Year,Month,Cases
0,Bhavanipur,Anaemia,Monsoon,2025,6,2
1,Bhavanipur,Anaemia,Monsoon,2025,7,1
2,Bhavanipur,Anaemia,Monsoon,2025,9,2
3,Bhavanipur,Anaemia,Monsoon,2026,6,1
4,Bhavanipur,Anaemia,Post-Monsoon,2025,10,2


In [26]:
label_encoders = {}

for column in ["Village", "Disease", "Season"]:

    encoder = LabelEncoder()

    cases[column] = encoder.fit_transform(
        cases[column]
    )

    label_encoders[column] = encoder

In [27]:
X = cases[
    [
        "Village",
        "Disease",
        "Season",
        "Year",
        "Month"
    ]
]

y = cases["Cases"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

RandomForestRegressor(n_estimators=300, random_state=42)

In [30]:
predictions = model.predict(X_test)

In [31]:
print("MAE :", mean_absolute_error(y_test, predictions))

print("RMSE :", np.sqrt(mean_squared_error(y_test, predictions)))

print("R2 :", r2_score(y_test, predictions))

MAE : 0.8393418259023356
RMSE : 1.04959237866841
R2 : -0.2607097327328718


In [32]:
importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

,Feature,Importance
0,Village,0.319590
1,Disease,0.314295
4,Month,0.235960
3,Year,0.078648
2,Season,0.051507


In [33]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    model,
    "disease_model.pkl"
)

joblib.dump(
    label_encoders,
    "label_encoders.pkl"
)

print("Model Saved Successfully")

Model Saved Successfully


In [34]:
sample = pd.DataFrame({

    "Village":[
        label_encoders["Village"].transform(
            ["Rampur"]
        )[0]
    ],

    "Disease":[
        label_encoders["Disease"].transform(
            ["Fever"]
        )[0]
    ],

    "Season":[
        label_encoders["Season"].transform(
            ["Winter"]
        )[0]
    ],

    "Year":[2026],

    "Month":[7]
})

prediction = model.predict(sample)

print("Predicted Cases :", round(prediction[0]))

Predicted Cases : 2


In [36]:
forecast = sample.copy()

forecast["Predicted_Cases"] = round(prediction[0])

forecast["Village"] = label_encoders["Village"].inverse_transform(
    forecast["Village"]
)

forecast["Disease"] = label_encoders["Disease"].inverse_transform(
    forecast["Disease"]
)

forecast["Season"] = label_encoders["Season"].inverse_transform(
    forecast["Season"]
)

os.makedirs("output", exist_ok=True)

forecast.to_csv(
    "disease_forecast.csv",
    index=False
)

forecast

,Village,Disease,Season,Year,Month,Predicted_Cases
0,Rampur,Fever,Winter,2026,7,2
